# 90｜指定路徑合併：新版優先、備份補齊

新版 fair-reference 結果優先；缺少的 model × dataset 才用備份1的最後一次匯入結果補齊。輸出會保留來源與公平排名資格。


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
ROOT=Path('/content/drive/MyDrive/姿態判定')
OUTPUT_ROOT=ROOT/'03_model_raw_outputs'
LEGACY_ROOT=ROOT/'04_merged_results/legacy_import_20260808_015836'
DATASET_IDS=('coco_singleperson_val2017','syrip_test100')
MODEL_IDS=('mediapipe_pose_heavy','movenet_thunder','rtmpose_m','yolo11x_pose','sapiens_1b','openpose_coco18')
EXPECTED={'batch_size':1,'warmup_images':10,'precision':'fp32','amp':False,'compile':False,'quantization':False,'tensor_rt':False,'rankable_across_models':True}
def canonical_protocol(protocol):
    p=dict(protocol or {})
    if 'tensorrt' in p and 'tensor_rt' not in p: p['tensor_rt']=p.pop('tensorrt')
    return {k:p.get(k) for k in EXPECTED}
print('✅ 新版輸出：',OUTPUT_ROOT)
print('✅ 備份補齊：',LEGACY_ROOT)


In [ ]:
import json,csv,datetime,hashlib
import pandas as pd
def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
def as_number(v):
    try: return float(v)
    except (TypeError,ValueError): return None
def value(row,names,default=None):
    for name in names:
        if name in row and pd.notna(row[name]): return row[name]
    return default
# A. 指定正式輸出路徑；同一 model × dataset 取最新完整 fair run。
fresh={}; diagnostics=[]
for model_id in MODEL_IDS:
    for dataset_id in DATASET_IDS:
        run_root=OUTPUT_ROOT/model_id/dataset_id/'fair_reference'
        manifests=sorted(run_root.glob('*/run_manifest.json')) if run_root.is_dir() else []
        if not manifests:
            diagnostics.append({'model_id':model_id,'dataset_id':dataset_id,'status':'missing_fair_run','path':str(run_root)})
            continue
        for p in manifests:
            try: m=json.loads(p.read_text(encoding='utf-8'))
            except Exception as e:
                diagnostics.append({'model_id':model_id,'dataset_id':dataset_id,'status':'invalid_manifest','path':str(p),'detail':str(e)}); continue
            valid=(m.get('profile_id',m.get('profile')) in ('fair_reference','fair_reference_v1') and canonical_protocol(m.get('protocol'))==EXPECTED and m.get('timing_comparability') is not False and int(m.get('counts',{}).get('input',-1)) in (921,100))
            if not valid:
                diagnostics.append({'model_id':model_id,'dataset_id':dataset_id,'status':'excluded_manifest','path':str(p),'detail':f"profile={m.get('profile_id',m.get('profile'))!r}"}); continue
            key=(model_id,dataset_id)
            if key not in fresh or m.get('run_id','')>fresh[key]['run_id']: fresh[key]={**m,'_path':str(p)}
# B. 讀取備份1最後一次匯入的 summary；僅補齊 fresh 未涵蓋的 key。
legacy_rows=[]; legacy_path=LEGACY_ROOT/'summary_metrics.csv'
if legacy_path.is_file():
    legacy_df=pd.read_csv(legacy_path)
    for _,r in legacy_df.iterrows():
        row=r.to_dict(); model_id=str(value(row,['model_id','model','model_name'],'')); dataset_id=str(value(row,['dataset_id','dataset','dataset_name'],''))
        if model_id and dataset_id in DATASET_IDS:
            legacy_rows.append({'run_id':str(value(row,['run_id'],'legacy_import_20260808_015836')),'model_id':model_id,'dataset_id':dataset_id,'images':as_number(value(row,['images','image_count','n_images'])),'coverage':as_number(value(row,['coverage','image_detection_coverage','match_recall'])),'oks_e2e':as_number(value(row,['oks_e2e','OKS_mean','paper_OKS_mean'])),'pck10_e2e':as_number(value(row,['pck10_e2e','PCK@0.10','PCK10','pck10'])),'oks_conditional':as_number(value(row,['oks_conditional','conditional_oks'])),'pck10_conditional':as_number(value(row,['pck10_conditional','conditional_pck10'])),'latency_mean_ms':as_number(value(row,['latency_mean_ms','mean_latency_ms','mean_pose_ms'])),'latency_median_ms':as_number(value(row,['latency_median_ms','median_latency_ms','median_pose_ms'])),'latency_p95_ms':as_number(value(row,['latency_p95_ms','p95_latency_ms'])),'timing_scope':'legacy_import','handoff_hash':None,'source_kind':'legacy_backup','fair_ranking_eligible':False,'timing_comparability':False,'source_path':str(legacy_path)})
else:
    diagnostics.append({'model_id':'*','dataset_id':'*','status':'missing_legacy_summary','path':str(legacy_path)})
# C. 先寫新資料，再由舊資料補缺；絕不以舊資料覆蓋新資料。
rows_by_key={}
for key,m in fresh.items():
    t=m.get('timing',{}); l=t.get('latency_ms',{})
    rows_by_key[key]={'run_id':m['run_id'],'model_id':m['model_id'],'dataset_id':m['dataset_id'],'images':t.get('images',m.get('counts',{}).get('input')),'coverage':t.get('coverage'),'oks_e2e':t.get('oks_e2e'),'pck10_e2e':t.get('pck10_e2e'),'oks_conditional':t.get('oks_conditional'),'pck10_conditional':t.get('pck10_conditional'),'latency_mean_ms':l.get('mean'),'latency_median_ms':l.get('median'),'latency_p95_ms':l.get('p95'),'timing_scope':m.get('timing_scope'),'handoff_hash':m.get('data_handoff_hash'),'source_kind':'fair_reference','fair_ranking_eligible':True,'timing_comparability':True,'source_path':m['_path']}
for row in legacy_rows: rows_by_key.setdefault((row['model_id'],row['dataset_id']),row)
rows=sorted(rows_by_key.values(),key=lambda r:(r['dataset_id'],r['model_id']))
if not rows:
    display(pd.DataFrame(diagnostics))
    raise RuntimeError('新版結果與指定備份 summary 都沒有可用資料。')
missing=[{'model_id':m,'dataset_id':d} for m in MODEL_IDS for d in DATASET_IDS if (m,d) not in rows_by_key]
stamp=datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ'); out=ROOT/'04_merged_results'/f'merged_{stamp}'; out.mkdir(parents=True,exist_ok=False)
with (out/'benchmark_table.csv').open('w',newline='',encoding='utf-8-sig') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
pd.DataFrame(diagnostics).to_csv(out/'merge_diagnostics.csv',index=False,encoding='utf-8-sig')
manifest={'kind':'hybrid_new_plus_legacy_backup','fresh_rows':sum(r['source_kind']=='fair_reference' for r in rows),'legacy_rows':sum(r['source_kind']=='legacy_backup' for r in rows),'missing_model_dataset':missing,'table_sha256':sha256(out/'benchmark_table.csv'),'fair_latency_rule':'只可使用 fair_ranking_eligible=True 的列做跨模型延遲排名；legacy_backup 僅供歷史準確度與回查。'}
(out/'merge_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(ROOT/'04_merged_results/latest.json').write_text(json.dumps({'path':str(out.relative_to(ROOT)),'kind':'hybrid_new_plus_legacy_backup','timing_comparability':False,'fair_ranking_eligible':False},ensure_ascii=False,indent=2),encoding='utf-8')
print(f"✅ 合併完成：新資料 {manifest['fresh_rows']} 筆，備份補齊 {manifest['legacy_rows']} 筆；仍缺 {len(missing)} 筆。")
display(pd.DataFrame(rows)); display(pd.DataFrame(missing))
